# Install Dependecies

In [1]:
%%capture
%pip install -q "transformers>=5,<6"
%pip install matplotlib
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [30]:
import numpy as np
import pandas as pd
import random
import torch
from collections import Counter
from datasets import load_dataset
from torch import nn
from torch.utils.data import DataLoader
from tqdm.std import tqdm
from transformers import AutoTokenizer
from transformers import AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
import ipynbname
from pathlib import Path
import time

In [13]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/nlp/msc-nlp-2026/project_notebooks


# Import datasets

In [14]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [15]:
LANGUAGES = ["ar", "ko", "te"]

In [16]:
# select languages
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# Tokeniser

In [24]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 5 Week 39: Learned Answerability
Implement two classifiers that receive both question and context:
1. an interpretable learned baseline, such as logistic regression using bag-ofwords counts, overlap, length or translation-derived features; and
2. a recurrent neural classifier using learned or pretrained representations.

Complete all of the following:
* motivate the representations and the way question and context information interact;
* evaluate both models under the shared protocol and compare them with the Week 36 baselines;
* manually inspect at least ten validation errors, including false positives and false negatives from all three languages where available, create categories for recurring failure types, and report category counts with representative examples; and
* if you use the Week 38 language-model score as a feature or diagnostic, explain why it should be informative.


In [18]:
# nlp_project_w36.ipynb
def evaluate_answerability(df, pred_column):
    rows = []
    for lang in ["all"] + LANGUAGES:
        part = df if lang == "all" else df[df["lang"] == lang]
        report = classification_report(part["answerable"], part[pred_column], labels=[True, False],
                                       target_names=["answerable", "unanswerable"], zero_division=0, output_dict=True)
        rows.append({
            "lang": lang,
            "n": len(part),
            "n_unans": int(report["unanswerable"]["support"]),
            "ans_p": report["answerable"]["precision"],
            "ans_r": report["answerable"]["recall"],
            "ans_f1": report["answerable"]["f1-score"],
            "unans_p": report["unanswerable"]["precision"],
            "unans_r": report["unanswerable"]["recall"],
            "unans_f1": report["unanswerable"]["f1-score"],
            "macro_f1": report["macro avg"]["f1-score"],
            "accuracy": report["accuracy"],
        })
    return pd.DataFrame(rows).round(3)

### Week 36 baselines

In [20]:
# nlp_project_w36.ipynb
majority_class = df_train["answerable"].mode()[0]

In [21]:
PUNCTUATION_TOKENS = {"▁", "?", "▁?", "؟", "▁؟"}

In [22]:
def rule_answerable(question, context):
    question_tokens = set(tokenizer.tokenize(question)) - PUNCTUATION_TOKENS
    context_tokens = set(tokenizer.tokenize(context))
    return len(question_tokens & context_tokens) > 0

In [25]:
# val
df_val["pred_majority"] = majority_class
df_val["pred_rule"] = [rule_answerable(question, context) for question, context in zip(df_val["question"], df_val["context"])]
display(evaluate_answerability(df_val, "pred_majority"))
display(evaluate_answerability(df_val, "pred_rule"))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (587 > 512). Running this sequence through the model will result in indexing errors


,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,1155,164,0.858,1.0,0.924,0.0,0.0,0.0,0.462,0.858
1,ar,415,52,0.875,1.0,0.933,0.0,0.0,0.0,0.467,0.875
2,ko,356,19,0.947,1.0,0.973,0.0,0.0,0.0,0.486,0.947
3,te,384,93,0.758,1.0,0.862,0.0,0.0,0.0,0.431,0.758


,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,1155,164,0.792,0.038,0.073,0.139,0.939,0.242,0.158,0.166
1,ar,415,52,1.000,0.022,0.043,0.128,1.000,0.227,0.135,0.145
2,ko,356,19,0.842,0.047,0.090,0.047,0.842,0.090,0.090,0.090
3,te,384,93,0.667,0.048,0.090,0.237,0.925,0.377,0.233,0.260


## 1. Interpretable baseline
Such as logistic regression using bag-ofwords counts, overlap, length or translation-derived features

In [26]:
# claude
def answerability_features(question, context, lang):
    question_pieces = [piece for piece in tokenizer.tokenize(question) if piece not in PUNCTUATION_TOKENS]
    context_pieces = set(tokenizer.tokenize(context))
    overlap = [piece for piece in question_pieces if piece in context_pieces]
    features = {
        "n_overlap": len(overlap),
        "share_overlap": len(overlap) / max(len(question_pieces), 1),
        "no_overlap": len(overlap) == 0,
        "log_q_len": np.log1p(len(question_pieces)),
        "log_c_len": np.log1p(len(context_pieces)),
        f"lang={lang}": 1.0,
    }
    for piece in question_pieces:
        features[f"q={piece}"] = 1.0
    for piece in overlap:
        features[f"overlap={piece}"] = 1.0
    return features

In [27]:
# train
train_features = [answerability_features(question, context, lang) for question, context, lang in zip(df_train["question"], df_train["context"], df_train["lang"])]

In [28]:
# val
val_features = [answerability_features(question, context, lang) for question, context, lang in zip(df_val["question"], df_val["context"], df_val["lang"])]

In [32]:
vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(train_features)
X_val = vectorizer.transform(val_features)

lr_classifier = LogisticRegression(max_iter=1000, class_weight="balanced")
start = time.time()
lr_classifier.fit(X_train, df_train["answerable"])
print(f"training time: {time.time() - start:.1f}s | examples: {X_train.shape[0]} | features: {X_train.shape[1]}")

training time: 0.1s | examples: 6335 | features: 7696


In [33]:
# train
df_train["pred_lr"] = lr_classifier.predict(X_train)
evaluate_answerability(df_train, "pred_lr")

,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,6335,363,1.0,0.986,0.993,0.816,1.0,0.899,0.946,0.987
1,ar,2558,255,1.0,0.970,0.985,0.789,1.0,0.882,0.934,0.973
2,ko,2422,63,1.0,0.997,0.999,0.900,1.0,0.947,0.973,0.997
3,te,1355,45,1.0,0.995,0.997,0.865,1.0,0.928,0.963,0.995


In [34]:
# val
df_val["pred_lr"] = lr_classifier.predict(X_val)
evaluate_answerability(df_val, "pred_lr")

,lang,n,n_unans,ans_p,ans_r,ans_f1,unans_p,unans_r,unans_f1,macro_f1,accuracy
0,all,1155,164,0.961,0.941,0.951,0.685,0.768,0.724,0.838,0.917
1,ar,415,52,0.994,0.983,0.989,0.893,0.962,0.926,0.957,0.981
2,ko,356,19,0.971,1.000,0.985,1.000,0.474,0.643,0.814,0.972
3,te,384,93,0.902,0.821,0.860,0.563,0.720,0.632,0.746,0.797


### Most informative features
positive weights -> answerable \
negative weights -> unanswerable.

In [35]:
# claude
weights = pd.Series(lr_classifier.coef_[0], index=vectorizer.get_feature_names_out()).sort_values()
display(pd.DataFrame({"unanswerable": weights.head(15).index, "weight": weights.head(15).values}).round(2))
display(pd.DataFrame({"answerable": weights.tail(15)[::-1].index, "weight": weights.tail(15)[::-1].values}).round(2))

,unanswerable,weight
0,q=▁هل,-7.02
1,q=컨,-2.06
2,q=▁ముఖ్యమంత్రి,-1.98
3,q=▁베이,-1.94
4,q=▁علوم,-1.94
5,q=▁తమిళనాడు,-1.89
6,q=룰,-1.76
7,q=▁있을까,-1.70
8,q=దేశం,-1.68
9,q=ష్,-1.66


,answerable,weight
0,q=▁언제,2.29
1,q=▁몇,2.01
2,q=▁어디,2.00
3,q=▁무엇인가,1.87
4,q=▁어느,1.64
5,q=▁누구,1.56
6,q=▁가장,1.52
7,q=تى,1.38
8,q=▁누가,1.21
9,q=▁얼마나,1.15


## recurrent neural classifier 
using learned or pretrained representations

In [36]:
MAX_LEN = 256
BATCH_SIZE = 32
LSTM_DIM = 128
DROPOUT = 0.3
LEARNING_RATE = 1e-3
N_EPOCHS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [37]:
# claude
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

### Encode pairs and build the vocabulary

In [38]:
def encode_pairs(df):
    return tokenizer(df["question"].tolist(), df["context"].tolist(), truncation="only_second", max_length=MAX_LEN)["input_ids"]

In [39]:
train_ids = encode_pairs(df_train)
val_ids = encode_pairs(df_val)
print("truncated (train):", np.mean([len(ids) == MAX_LEN for ids in train_ids]).round(3))

truncated (train): 0.113


In [41]:
# 0 = <pad>, 1 = <unk>, then every XLM-R piece seen in the training split
vocabulary = sorted({piece_id for ids in train_ids for piece_id in ids})
PIECE_TO_INDEX = {piece_id: index + 2 for index, piece_id in enumerate(vocabulary)}

In [42]:
def to_indices(ids):
    return [PIECE_TO_INDEX.get(piece_id, 1) for piece_id in ids]

In [43]:
train_indices = [to_indices(ids) for ids in train_ids]
val_indices = [to_indices(ids) for ids in val_ids]
print("vocabulary:", len(vocabulary), "pieces + <pad> + <unk>")
print("val <unk> share:", round(np.mean([index == 1 for indices in val_indices for index in indices]), 4))

vocabulary: 31042 pieces + <pad> + <unk>
val <unk> share: 0.0088


In [47]:
# pretrained XLM-R input embeddings for the vocabulary
xlmr = AutoModel.from_pretrained("xlm-roberta-base")
xlmr_embeddings = xlmr.get_input_embeddings().weight.detach()
embedding_matrix = torch.cat([
    torch.zeros(1, xlmr_embeddings.shape[1]),
    xlmr_embeddings[tokenizer.unk_token_id].unsqueeze(0),
    xlmr_embeddings[vocabulary],
])

print("embedding matrix:", tuple(embedding_matrix.shape))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding matrix: (31044, 768)


### Data loaders